In [1]:
# Add the parent directory to Python path for development
import sys
import os
sys.path.insert(0, os.path.abspath('..'))


## Step 1: Build a Quantum Hamiltonian

Let's start by creating a simple Heisenberg chain Hamiltonian.


In [2]:
from shadowgpt.physics import ham_tf_ising
ham = ham_tf_ising(n=10, gs=[0.43], bc="periodic")
ham

-0.43 X9 - 0.57 Z8 Z9 - 0.43 X8 - 0.57 Z7 Z8 - 0.43 X7 - 0.57 Z6 Z7 - 0.43 X6 - 0.57 Z5 Z6 - 0.43 X5 - 0.57 Z4 Z5 - 0.43 X4 - 0.57 Z3 Z4 - 0.43 X3 - 0.57 Z2 Z3 - 0.43 X2 - 0.57 Z1 Z2 - 0.43 X1 - 0.57 Z0 Z9 - 0.57 Z0 Z1 - 0.43 X0

## Step 2: Exact Diagonalization (ED)

Exact diagonalization is a method to find the ground state by diagonalizing the full Hamiltonian matrix. It's exact but limited to small systems due to exponential scaling with system size.

**When to use ED:**
- Small systems (typically ≤ 20 qubits)
- When you need exact results
- For benchmarking other methods

**Key features:**
- Returns the exact ground state vector
- Computationally expensive for large systems
- Memory scales as 2^n where n is the number of qubits


In [4]:
from shadowgpt.physics import EDSolver, EDConfig
import numpy as np

# Create ED solver with default configuration
ed_solver = EDSolver()
# Solve for ground state
energy, state = ed_solver.solve(ham)
print(f"Ground energy: {energy:.6f}")
print(f"Ground state norm: {np.linalg.norm(state):.6f}")


Ground energy: -6.549615
Ground state norm: 1.000000


In [5]:
from pyclifford import pauli
ed_solver.expectation_value(pauli({0:'Z', 3:'Z'}))

(0.8019265817981612+0j)

In [7]:
[ed_solver.entropy(i) for i in range(1,ed_solver.ham.N)]

[0.8677584421671662,
 0.9807921843084217,
 1.0213293299706665,
 1.0379013809733135,
 1.0425230594500894,
 1.0379013809733115,
 1.0213293299706658,
 0.9807921843084205,
 0.8677584421671658]

## Step 3: Density Matrix Renormalization Group (DMRG)

DMRG is a variational method that finds ground states by optimizing a Matrix Product State (MPS) representation. It's much more efficient than ED for 1D systems and can handle larger system sizes.

**When to use DMRG:**
- 1D quantum systems
- Larger systems (hundreds of sites possible)
- When you need good approximations to ground states
- Systems with limited entanglement

**Key features:**
- Returns a Matrix Product State (MPS) representation
- Scalable to much larger systems than ED
- Configurable bond dimensions and convergence criteria
- Shows convergence information and energy history


In [8]:
from shadowgpt.physics import DMRGSolver, DMRGConfig

# Create DMRG solver with custom configuration
dmrg_config = DMRGConfig(
    bond_dims=[10, 20, 30, 40, 50],
    cutoffs=1e-09,
    bsz=2,
    which='SA',
    tol=0.0001,
    max_sweeps=10,
    verbosity=1
)
dmrg_solver = DMRGSolver(dmrg_config)

# Solve for ground state
energy, state = dmrg_solver.solve(ham)
print(f"Ground energy: {energy:.6f}")
print(f"Number of sweeps: {len(dmrg_solver.energies)}")
print(f"Converged: {dmrg_solver.converged}")
print(f"Energy history: {dmrg_solver.energies}")

# Show the MPS structure (bond dimensions are displayed here)
print("\nMPS structure:")
state.show()

1, R, max_bond=(10/10), cutoff:1e-09


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/cotengra/hyperoptimizers/hyper.py:55: UserWarning: Couldn't find `optuna`, `cmaes`, or `nevergrad` so will use completely random sampling in place of hyper-optimization. It is recommended to install one of these libraries for higher quality contraction paths.
  warnings.warn(
100%|#############################################| 9/9 [00:00<00:00, 51.18it/s]

Energy: (-6.548269059528128+2.5306387547088652e-15j) ... not converged.
2, R, max_bond=(10/20), cutoff:1e-09



100%|#############################################| 9/9 [00:00<00:00, 56.59it/s]

Energy: (-6.549601182964825+1.1079821180284858e-15j) ... not converged.
3, R, max_bond=(17/30), cutoff:1e-09



100%|#############################################| 9/9 [00:00<00:00, 76.87it/s]

Energy: (-6.549614066048406-1.351849369515873e-16j) ... converged!
Ground energy: -6.549614-0.000000j
Number of sweeps: 3
Converged: True
Energy history: [(-6.548269059528128+2.5306387547088652e-15j), (-6.549601182964825+1.1079821180284858e-15j), (-6.549614066048406-1.351849369515873e-16j)]

MPS structure:
 2 4 8 12 15 13 8 4 2 
>─>─>─>──>──>──>─>─>─●
│ │ │ │  │  │  │ │ │ │


In [9]:
from pyclifford import pauli
from shadowgpt.physics import Operator

dmrg_solver.expectation_value(Operator(pauli({0:'Z', 3:'Z'})))


0.8019325113949868

In [10]:
[dmrg_solver.entropy(i) for i in range(1,dmrg_solver.ham.N)]

[0.8675878731415585,
 0.9805880381169361,
 1.021113080391842,
 1.0376820360102224,
 1.0423066500747078,
 1.0376959703790676,
 1.0211413051026654,
 0.9806029334856822,
 0.8675876512375664]